In [ ]:
import os
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from tsgettoolbox import tsgettoolbox as tsget
from wdmtoolbox import wdmtoolbox as wdm

from cloud_cover_timeseries_from_solar import cloud_cover_timeseries_from_solar

import pandas as pd


CONSTITUENTS = ["ATEMP", "PRECIP", "SOLR", "WIND", "CLOU"]

GLDAS_CONSTITUENT_DETAILS = {
    "PRECIP": ["Precipitation", "mm", "GLDAS2:GLDAS_NOAH025_3H_2_1_Rainf_f_tavg", "kg/m^2/s"],
    "ATEMP": ["Air Temperature", "Celsius", "GLDAS2:GLDAS_NOAH025_3H_2_1_Tair_f_inst", "K"],
    "SOLR": ["Shortwave radiation flux", "Langleys", "GLDAS2:GLDAS_NOAH025_3H_2_1_SWdown_f_tavg", "W m-2"],
    "WIND": ["Wind Speed", "m s-1", "GLDAS2:GLDAS_NOAH025_3H_2_1_Wind_f_inst", "m s-1"],
    "CLOU": ["Cloud Cover Estimate", "Cloud Fraction", "GLDAS2:GLDAS_NOAH025_3H_2_1_SWdown_f_tavg", "W m-2"],
}

NLDAS_CONSTITUENT_DETAILS = {
    "PRECIP": ["Precipitation", "mm", "NLDAS:NLDAS_FORA0125_H_2_0_Rainf", "mm"],
    "ATEMP": ["Air Temperature", "Celsius", "NLDAS:NLDAS_FORA0125_H_2_0_Tair", "K"],
    "SOLR": ["Shortwave radiation flux", "Langleys", "NLDAS:NLDAS_FORA0125_H_2_0_SWdown", "W m-2"],
    "WIND": [
        "Wind Speed",
        "m s-1",
        "NLDAS:NLDAS_FORA0125_H_2_0_Wind_N",
        "NLDAS:NLDAS_FORA0125_H_2_0_Wind_E",
        "m s-1",
    ],
    "CLOU": ["Cloud Cover Estimate", "Cloud Fraction", "NLDAS:NLDAS_FORA0125_H_2_0_SWdown", "W m-2"],
}

US_BOUNDS = {
    "north": 49.3457868,
    "south": 24.7433195,
    "west": -124.7844079,
    "east": -66.9513812,
}


@dataclass(frozen=True)
class Station:
    name: str
    latitude: float
    longitude: float
    timezone_adjustment: float


# Option A: inline stations
stations = [
    Station("CentralPark", 40.785091, -73.968285, -5),
    Station("GoldenGate", 37.819929, -122.478255, -8),
    Station("Millennium", 41.882702, -87.619392, -6),
]

# Option B: load from CSV instead
# station_df = pd.read_csv("station_example.csv")
# stations = [
#     Station(
#         row["StationName"],
#         float(row["Latitude"]),
#         float(row["Longitude"]),
#         float(row["TimeZoneAdjustment"])
#     )
#     for _, row in station_df.iterrows()
# ]

start_date = "2025-01-01"
end_date = "2025-01-03"

WDMFileName = "MetData_clean.wdm"
LogFileName = "MetLog_clean.txt"

def validate_lat_lon(latitude, longitude):
    return -90 <= float(latitude) <= 90 and -180 <= float(longitude) <= 180


def in_contiguous_usa(station):
    return (
        US_BOUNDS["south"] < station.latitude < US_BOUNDS["north"]
        and US_BOUNDS["west"] < station.longitude < US_BOUNDS["east"]
    )


def get_catalog(station):
    if in_contiguous_usa(station):
        return NLDAS_CONSTITUENT_DETAILS, 1, "NLDAS"
    return GLDAS_CONSTITUENT_DETAILS, 3, "GLDAS"


def normalize_series(data, name):
    if isinstance(data, pd.DataFrame):
        if data.shape[1] == 0:
            raise ValueError("Retrieved DataFrame has no columns.")
        series = data.iloc[:, 0].copy()
    elif isinstance(data, pd.Series):
        series = data.copy()
    else:
        raise TypeError(f"Expected pandas Series/DataFrame, got {type(data).__name__}")

    series.index = pd.to_datetime(series.index)
    series = pd.to_numeric(series, errors="coerce")
    series = series.dropna().sort_index()
    series.name = name
    return series


def convertunitforHSPF(constituent, series, ldas_var):
    series = normalize_series(series, constituent)

    if constituent == "ATEMP":
        series = series - 273.15
    elif constituent == "SOLR":
        series = series * 0.0864
    elif constituent == "PRECIP" and "GLDAS" in ldas_var:
        series = series * 3600 * 3

    series.name = constituent
    return series


def compute_cloud_cover(solr_series, lat_deg):
    solr_series = normalize_series(solr_series, "SOLR")
    solr_series = convertunitforHSPF("SOLR", solr_series, "SOLR")

    clou_df = cloud_cover_timeseries_from_solar(
        solr_values=solr_series.tolist(),
        dates=solr_series.index.to_pydatetime().tolist(),
        lat_deg=lat_deg,
    )

    clou_series = normalize_series(clou_df, "CLOU")
    clou_series.name = "CLOU"
    return clou_series


def fetch_ldas_series(station, variable_name, start_date, end_date):
    result = tsget.ldas(
        lat=station.latitude,
        lon=station.longitude,
        variables=variable_name,
        startDate=start_date,
        endDate=end_date,
    )
    return normalize_series(result, variable_name)


def prepare_wdm_input(series, constituent):
    series = normalize_series(series, constituent)
    if series.empty:
        raise ValueError(f"No data available to write for constituent {constituent}.")
    return series.to_frame(name=constituent)

def convertunitforHSPF(constituent, df, LDAS_var):
    """Unit conversion only. Always return a pandas Series with DateTimeIndex."""
    if isinstance(df, pd.DataFrame):
        df = df.iloc[:, 0]

    df = df.copy()
    df.index = pd.to_datetime(df.index)
    df = pd.to_numeric(df, errors="coerce")
    df = df.dropna()

    if constituent == "ATEMP":
        df = df - 273.15   # K to C

    elif constituent == "SOLR":
        df = df * 0.0864   # W/m^2 to langleys for your workflow
        df.name = "SOLR"

    elif constituent == "PRECIP" and "GLDAS" in LDAS_var:
        df = df * 3600 * 3

    elif constituent == "PRECIP" and "NLDAS" in LDAS_var:
        df = df

    return df

In [ ]:
def build_clou_from_solr(solr_series, lat_deg):
    """Compute CLOU from SOLR and return a pandas Series."""
    solr_series = convertunitforHSPF("SOLR", solr_series, "SOLR")

    clou_df = cloud_cover_timeseries_from_solar(
        solr_values=solr_series.tolist(),
        dates=solr_series.index.to_pydatetime().tolist(),
        lat_deg=lat_deg
    )

    clou_series = clou_df["CLOU"].copy()
    clou_series.index = pd.to_datetime(clou_series.index)
    clou_series.name = "CLOU"
    return clou_series.dropna()

In [ ]:
if const == "WIND":
...
elif const == "CLOU":
...
else:
...

In [ ]:
if station[1] < UStop and station[1] > USbottom and station[2] > USleft and station[2] < USright:
    # NLDAS
    if const == "WIND":
        wind_n_var = NLDAS_ConstituentDetails["WIND"][2]
        wind_e_var = NLDAS_ConstituentDetails["WIND"][3]
        TimeStep = 1

        df_n = tsget.ldas(
            lat=station[1], lon=station[2],
            variables=wind_n_var,
            startDate=start_date, endDate=end_date
        )
        df_e = tsget.ldas(
            lat=station[1], lon=station[2],
            variables=wind_e_var,
            startDate=start_date, endDate=end_date
        )

        wind_speed = np.sqrt(df_n.iloc[:, 0] ** 2 + df_e.iloc[:, 0] ** 2)
        wind_speed.name = "WIND"
        df = wind_speed.dropna()
        LDAS_variable = f"{wind_n_var} + {wind_e_var}"
        column_name = "WIND"

    elif const == "CLOU":
        solar_n_var = NLDAS_ConstituentDetails["CLOU"][2]
        TimeStep = 1

        raw_df = tsget.ldas(
            lat=station[1], lon=station[2],
            variables=solar_n_var,
            startDate=start_date, endDate=end_date
        )

        solr_series = raw_df.iloc[:, 0].dropna()
        df = build_clou_from_solr(solr_series, lat_deg)
        LDAS_variable = solar_n_var
        column_name = "CLOU"

    else:
        LDAS_variable = NLDAS_ConstituentDetails[const][2]
        TimeStep = 1

        raw_df = tsget.ldas(
            lat=station[1], lon=station[2],
            variables=LDAS_variable,
            startDate=start_date, endDate=end_date
        )

        df = raw_df.iloc[:, 0]
        df = convertunitforHSPF(const, df, LDAS_variable)
        column_name = const

In [ ]:
if isinstance(df, pd.Series):
    df = df.to_frame(name=df.name if df.name else const)
elif isinstance(df, pd.DataFrame):
    if df.shape[1] != 1:
        raise ValueError("wdm.csvtowdm requires a one-column DataFrame.")
else:
    raise TypeError(f"Expected pandas Series/DataFrame, got {type(df).__name__}")

df.index = pd.to_datetime(df.index)
df = df.sort_index()
df.iloc[:, 0] = pd.to_numeric(df.iloc[:, 0], errors="coerce")
df = df.dropna()
wdm.csvtowdm(WDMFileName, index, input_ts=df)

In [ ]:
station = stations[0]
constituent = "CLOU"

catalog, timestep_hours, dataset_name = get_catalog(station)

if constituent == "WIND" and dataset_name == "NLDAS":
    wind_n_var = catalog["WIND"][2]
    wind_e_var = catalog["WIND"][3]

    wind_n = fetch_ldas_series(station, wind_n_var, start_date, end_date)
    wind_e = fetch_ldas_series(station, wind_e_var, start_date, end_date)

    wind_n, wind_e = wind_n.align(wind_e, join="inner")
    series = np.sqrt(wind_n.pow(2) + wind_e.pow(2))
    series = normalize_series(series, "WIND")
    description = "Vector Wind Speed"

elif constituent == "CLOU":
    solar_var = catalog["CLOU"][2]
    solr_series = fetch_ldas_series(station, solar_var, start_date, end_date)
    series = compute_cloud_cover(solr_series, station.latitude)
    description = solar_var

else:
    variable_name = catalog[constituent][2]
    series = fetch_ldas_series(station, variable_name, start_date, end_date)
    series = convertunitforHSPF(constituent, series, variable_name)
    description = variable_name

display(series.head())
print(type(series))